# ADS-509 Assignment 3.1
## Word Embeddings

In this assignment, you will use the HackerNews dataset created in the Module 1 assignment o:
- Build static embeddings by training Word2Vec
- Build contextual embeddings with a sentence-transformer (BERT-family)
- Perform EDA on embeddings  
- Train a simple classifier (logistic regression) on each embedding type to predict a lightweight label from story titles

If you are not confident in the quality of your own dataset from Module 1, there is a clean dataset available for your use in the assignment repository on GitHub.

## General Assignment Instructions

These instructions are included in every assignment, to remind you of the coding standards for the class. Feel free to delete this cell after reading it.

Work through this notebook as if it were a worksheet, completing the code sections marked with **TODO** in the cells provided. Similarly, written questions will be marked by a "Q:" and will have a corresponding "A:" spot for you to fill in with your answers. **Make sure to answer every question marked with a Q: for full credit**.

Your code should be relatively easy-to-read, sensibly commented, and clean. Writing code is a messy process, so please be sure to edit your final submission. Remove any cells that are not needed or parts of cells that contain unnecessary code. Remove inessential import statements and make sure that all such statements are moved into the designated cell.

A .pdf of this notebook, with your completed code and written answers, is what you should submit in Canvas for full credit. **DO NOT SUBMIT A NEW NOTEBOOK FILE OR A RAW .PY FILE**. Submitting in a different format makes it difficult to grade your work, and students who have done this in the past inevitably miss some of the required work or written questions.

## Imports and Downloads

Once again we will use some datasets from the NLTK library, so we need to make sure that these are downloaded (you should have them from Module 2, but it never hurts to double check).

We will also be using the pre-trained embedding models (Word2Vec and SentenceTransformer) from Gensim.

In [3]:
import os, re, string, random, math, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ast

from tqdm import tqdm

# Text preprocessing
import nltk
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')
try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Embeddings
from gensim.models import Word2Vec
from sentence_transformers import SentenceTransformer

# ML
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

plt.rcParams['figure.figsize'] = (8,5)
plt.rcParams['figure.dpi'] = 120


## Load Data

Next we will load our dataset from Module 2 and double check that it is formatted correctly.

If you are uncertain about your own dataset, or if you don't pass the check below, feel free to use the dataset provided on Canvas.

In [4]:
DATA_PATH = 'data/module2/hn_comment_features.csv'  # TODO: Update the file path as needed

assert os.path.exists(DATA_PATH), f"Dataset not found at {DATA_PATH}. Update the path for your environment."

In [8]:
df = pd.read_csv(DATA_PATH)
print('Rows:', len(df))
expected_cols = {'story_id','title','comment_id','user','comment_text','text_norm','tokens_clean','sent_compound'}
missing = expected_cols - set(df.columns)
if missing:
    print('Warning: missing expected columns:', missing)

df["tokens_clean"] = df["tokens_clean"].apply(ast.literal_eval) # convert python list from a string
df.head()

Rows: 3607


,story_id,title,comment_id,user,comment_text,text_norm,tokens_clean,sent_compound
0,45116688,Claude Code: Now in Beta in Zed,45118137,unshavedyak,I want to try Zed but the Helix mode seems qui...,i want to try zed but the helix mode seems qui...,"[want, try, zed, helix, mode, seems, quite, yo...",0.8462
1,45116688,Claude Code: Now in Beta in Zed,45118867,Karrot_Kream,Helix seems to have good LSP support from what...,helix seems to have good lsp support from what...,"[helix, seems, good, lsp, support, tell, langu...",0.8225
2,45116688,Claude Code: Now in Beta in Zed,45118827,yes_but_no,If you are already familiar with Vim bindings ...,if you are already familiar with vim bindings ...,"[already, familiar, vim, bindings, helix, obje...",0.2944
3,45116688,Claude Code: Now in Beta in Zed,45118464,ppeetteerr,I love Zed and I'm glad you now have native su...,i love zed and i'm glad you now have native su...,"[love, zed, glad, native, support, claude, pre...",0.9231
4,45116688,Claude Code: Now in Beta in Zed,45119080,hajile,I was somewhat surprised to find that Zed stil...,i was somewhat surprised to find that zed stil...,"[somewhat, surprised, find, zed, still, way, a...",0.9119


In [9]:
if missing:
    print('Warning: missing expected columns:', missing)

## Create a label for classification

The dataset that we scraped from HackerNews doesn't have an obvious variable for use in a classification task (which we will need later). There are many ways that we could create such a label, but here we will use string matching to create some rough labels based on the words used in the article titles.

**Q**: Give at least two other ideas for how we could label our dataset for classification. Keep data balance in mind (i.e. the resulting dataset should not be completely dominated by one label) in a brief discussion of why that method would/wouldn't be a good choice.

**A**: One way we could label the data is based on the sentiment of each comment using the sent_compound score from Module 2. We could separate the comments into positive, neutral, and negative categories. This could be useful for predicting the overall sentiment of a comment based on its text. However, we would need to check how many comments fall into each category because if most of the comments have similar sentiment, the classes could become unbalanced.

Another way would be to label the stories based on how much engagement they received. For example, we could classify stories as high or low engagement based on the number of comments they received. We could use the median number of comments as the cutoff with stories above the median labeled as high engagement and those below it as low engagement. This would be a good option because using the median could help keep the two classes more evenly balanced.

In [11]:
# Build a simple label from the story title
def label_from_title(title):
    if not isinstance(title, str):
        return None
    t = title.strip().lower()
    if any(kw in t for kw in ['rust', 'python', 'sql', 'linux', 'windows', 'ios', 'c++', 'perl', 'pfp', 'java']):
        return 'programming'
    if any(kw in t for kw in ['google', 'facebook', 'meta', 'apple', 'amazon', 'microsoft']):
        return 'big-tech'
    if any(kw in t for kw in ['ai ', 'torch', 'llm', 'large language model', 'claude', 'gemini', 'copilot']):
        return 'ai'
    else:
        return 'other'
    return None

df['label'] = df['title'].apply(label_from_title)

print('Total rows for task:', len(df))
print(df['label'].value_counts())

df[['title','comment_text','label']].head(3)

Total rows for task: 3607
label
other          1569
ai             1119
big-tech        770
programming     149
Name: count, dtype: int64


,title,comment_text,label
0,Claude Code: Now in Beta in Zed,I want to try Zed but the Helix mode seems qui...,ai
1,Claude Code: Now in Beta in Zed,Helix seems to have good LSP support from what...,ai
2,Claude Code: Now in Beta in Zed,If you are already familiar with Vim bindings ...,ai


## Train Static Word Embedding Model

We will train a Word2Vec model to produce static word embeddings for our normalized text, which we will use later for data exploration and classification.

**TODO**:

Use the gensim Word2Vec class to train a static embedding model on our tokenized comment text. Check out the [documentation](https://tedboy.github.io/nlps/generated/generated/gensim.models.Word2Vec.html#gensim-models-word2vec) to assign the following settings (Hint: You might need to go into the class source code to find argument descriptions):

- Embedding size 100
- Sequence window size 5
- Limit to tokens that appear at least 3 times
- Skip-gram algorithm (rather than CBOW)
- Run training for 10 epochs
- If you have multiple CPUs available, set the number of workers to improve training speed

**Q**: Why do we call a model like Word2Vec a *static* word embedding model?

**A**: Word2Vec is called a static word embedding model because each word gets one fixed vector. The vector stays the same no matter how or where the word is used in a sentence.

**Q**: What is the difference between the CBOW and skip-gram algorithms?

**A**: CBOW uses the words around a target word to predict that word. Skip-gram does the opposite by using the target word to predict the words around it. In this assignment, we used skip-gram.

In [ ]:

sentences = df["tokens_clean"].tolist()

w2v = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=3,
    sg=1,
    epochs=10,
    workers=os.cpu_count()
)

w2v_vecs = w2v.wv
len(w2v_vecs), w2v_vecs.vector_size

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


(5196, 100)

The word embeddings that we just created can now be used to perform a semantic comparison of individual words/tokens in our vocabulary.

**TODO**:
- Choose a few words from our corpus vocabulary and print the 5 nearest neighbors using the Word2Vec.most_similar() function

**Q**: Are the nearest neighbors semantically similar to your chosen words? Think about how the Word2Vec model works and provide a 2-3 sentence description of why this comparison does/doesn't work for our dataset.

**A**: Most of the nearest neighbors are related to the chosen words. For example, ai is related to words like prompts, generated, and research, while code is related to tests and rust. Some results are less obvious because Word2Vec learns from how words are used together in our specific dataset rather than their exact definitions.


In [ ]:

probe_terms = ['ai', 'code', 'python']

for term in probe_terms:
    print(f"\nNearest neighbors for '{term}':")
    print(w2v_vecs.most_similar(term, topn=5))


Nearest neighbors for 'ai':
[('research', 0.7814264893531799), ('generated', 0.7514234781265259), ('hype', 0.7300251722335815), ('prompts', 0.7281234860420227), ('slop', 0.7277953028678894)]

Nearest neighbors for 'code':
[('tests', 0.7867076396942139), ('generate', 0.7775375843048096), ('agents', 0.7771809697151184), ('rust', 0.774882435798645), ('agent', 0.7737991213798523)]

Nearest neighbors for 'python':
[('lines', 0.983877956867218), ('integration', 0.9728183150291443), ('rust', 0.972113311290741), ('diffs', 0.9716328382492065), ('lets', 0.9695404171943665)]


## Build Document Embeddings 

One way to represent a document (comment) in our dataset is to aggregate all of the individual word embeddings by computing an average embedding or document vector. This vector can then be used to compare entire documents instead of individual words.

**TODO**:
- Average the word/token embeddings for each comment in our dataset to produce a single document embedding vector
- Compute the cosine similarity between the first comment's document embedding and all of the rest
- Print out the two most and least similar comments

**Q**: Describe your results. Does the cosine similarity of document/comment embeddings do a good job of identifying similar content?

**A**: The cosine similarity did a good job of finding comments with similar content. The comments with the highest similarity discussed similar topics, such as text editors and LSPs. The least similar comments were both flagged, so they did not contain enough text to make a meaningful comparison.

In [ ]:

def docvec_average(tokens, wv):
    vecs = [wv[t] for t in tokens if t in wv]
    if not vecs:
        return np.zeros(wv.vector_size, dtype=np.float32)
    return np.mean(vecs, axis=0)

df["w2v_embedding"] = list(np.vstack([docvec_average(toks, w2v_vecs) for toks in df['tokens_clean']]))


In [20]:

def docvec_similarity(target, docvecs):
    sims = cosine_similarity([target], list(docvecs))[0]
    return sims.tolist()

df["w2v_sim"] = [None] + list(
    docvec_similarity(df["w2v_embedding"][0], df["w2v_embedding"][1:])
)

In [22]:
print("Target Comment:")
print(df["comment_text"][0])

print("\nLowest Similarities:")
print(df.nsmallest(2, "w2v_sim")[["comment_text", "w2v_sim"]])

print("\nHighest similarities:")
print(df.nlargest(2, "w2v_sim")[["comment_text", "w2v_sim"]])


Target Comment:
I want to try Zed but the Helix mode seems quite young. Vim mode sounds good, but i just can't move away from Helix mode. (oh and of course, my own modifications to Helix's input config) My difficulty in finding editors that fit my desired input scheme kinda reminds me of the old pre-LSP days. Where you'd chose an editor based on it's language features. I wonder if we need some sort of common editor interface to allow these sort of text editing primitives to work in new editors, as it seems to be considerable friction. reply

Lowest Similarities:
     comment_text   w2v_sim
1177    [flagged]  0.000241
1245    [flagged]  0.000241

Highest similarities:
                                         comment_text   w2v_sim
87  Even not counting the LSP, for a text editor, ...  0.994894
90  Strange. I just opened the same project in Cur...  0.993979


## Build Contextual Embeddings with Sentence-Transformers

Another way to create a document-level embedding is to use a more sophisticated, pre-trained embedding model, like a Sentence Transformer (based on the BERT family architecture). These models use an *attention block* to create context-specific embeddings for a string of text, rather than simply averaging the static word embeddings as we did above.

**TODO**:
- Use the gensim SentenceTransformer class to load the  pre-trained 'sentence-transformers/all-MiniLM-L6-v2' model
- Use this model to create embeddings for our comment text (HINT: You don't need to normalize or tokenize text before feeding it into a transformer model)


In [23]:
model_name = 'sentence-transformers/all-MiniLM-L6-v2'
st_model = SentenceTransformer(model_name)

df["bert_embedding"] = list(
    st_model.encode(df["comment_text"].fillna("").tolist())
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12506.18it/s]


Now we will compute the same cosine similarities as we did with the Word2Vec embeddings above.

**Q**: How do the transformer-based embedding similarities compare to what you found above? Which method would you choose and why?

**A**: The transformer embeddings had lower similarity scores than Word2Vec. The highest Word2Vec similarities were around 0.99, while the highest transformer similarity was about 0.68. However, the transformer found a comment about Zed as the most similar to the target comment, which makes sense because the target also discusses Zed and text editors. The lowest similarities were negative and came from comments about unrelated topics. I would choose the transformer method because it considers the context and overall meaning of the comments instead of simply averaging individual word embeddings.

In [24]:
df["bert_sim"] = [None] + docvec_similarity(df["bert_embedding"][0], df["bert_embedding"][1:]) # create new column with None for self-similarity

In [25]:
print("Target Comment:")
print(df["comment_text"][0])

print("\nLowest Similarities:")
print(df.nsmallest(2, "bert_sim")[["comment_text", "bert_sim"]])

print("\nHighest similarities:")
print(df.nlargest(2, "bert_sim")[["comment_text", "bert_sim"]])

Target Comment:
I want to try Zed but the Helix mode seems quite young. Vim mode sounds good, but i just can't move away from Helix mode. (oh and of course, my own modifications to Helix's input config) My difficulty in finding editors that fit my desired input scheme kinda reminds me of the old pre-LSP days. Where you'd chose an editor based on it's language features. I wonder if we need some sort of common editor interface to allow these sort of text editing primitives to work in new editors, as it seems to be considerable friction. reply

Lowest Similarities:
                                           comment_text  bert_sim
3243  This is a civil case. The point is to remedy t... -0.186866
3092  they likely, and probably correctly, do not wa... -0.177253

Highest similarities:
                                          comment_text  bert_sim
152  zed extension is too bad to be used as a full-...  0.677772
19   What most of these comments are missing is the...  0.570953


In [28]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Train/test split (same split for both)
Xw_train, Xw_test, y_train, y_test = train_test_split(
    df['w2v_embedding'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

Xb_train, Xb_test, _, _ = train_test_split(
    df['bert_embedding'],
    df['label'],
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

# Standardize
sc_w2v = StandardScaler(with_mean=False)
Xw_train_s = sc_w2v.fit_transform(np.stack(Xw_train.to_numpy()))
Xw_test_s = sc_w2v.transform(np.stack(Xw_test.to_numpy()))

sc_bert = StandardScaler(with_mean=False)
Xb_train_s = sc_bert.fit_transform(np.stack(Xb_train.to_numpy()))
Xb_test_s = sc_bert.transform(np.stack(Xb_test.to_numpy()))

# Train logistic regression models with both embedding methods
clf_w2v = LogisticRegression(max_iter=1000, random_state=42)
clf_w2v.fit(Xw_train_s, y_train)
pred_w2v = clf_w2v.predict(Xw_test_s)

clf_bert = LogisticRegression(max_iter=1000, random_state=42)
clf_bert.fit(Xb_train_s, y_train)
pred_bert = clf_bert.predict(Xb_test_s)

# Print logistic regression model performance metrics
print("\n=== Word2Vec Avg → Logistic Regression ===\n")
print(classification_report(y_test, pred_w2v, digits=3))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_w2v))

print("\n=== Sentence-BERT → Logistic Regression ===\n")
print(classification_report(y_test, pred_bert, digits=3))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred_bert))


=== Word2Vec Avg → Logistic Regression ===

              precision    recall  f1-score   support

          ai      0.726     0.696     0.711       224
    big-tech      0.684     0.604     0.641       154
       other      0.693     0.799     0.743       314
 programming      0.556     0.167     0.256        30

    accuracy                          0.699       722
   macro avg      0.665     0.567     0.588       722
weighted avg      0.696     0.699     0.691       722

Confusion Matrix:
 [[156   7  59   2]
 [ 23  93  37   1]
 [ 31  31 251   1]
 [  5   5  15   5]]

=== Sentence-BERT → Logistic Regression ===

              precision    recall  f1-score   support

          ai      0.749     0.746     0.747       224
    big-tech      0.698     0.734     0.715       154
       other      0.787     0.777     0.782       314
 programming      0.370     0.333     0.351        30

    accuracy                          0.740       722
   macro avg      0.651     0.647     0.649       72

In [29]:
print(classification_report(y_test, pred_bert, digits=3))

              precision    recall  f1-score   support

          ai      0.749     0.746     0.747       224
    big-tech      0.698     0.734     0.715       154
       other      0.787     0.777     0.782       314
 programming      0.370     0.333     0.351        30

    accuracy                          0.740       722
   macro avg      0.651     0.647     0.649       722
weighted avg      0.739     0.740     0.739       722



## Embedding-based classification

Finally, we will explore how well the document embeddings perform on the classification task of predicting our comment category labels. We will use the two types of embeddings for input to two different logistic regression models, and then compare the performance.

**TODO**:

- Use the sklearn LogisticRegression class to predict our comment labels using both embedding methods

**Q**: How do the two embedding methods perform? Use the results from the model metrics and the confusion matrices to discuss and compare.

**A**: Sentence-BERT performed better overall than Word2Vec. The Word2Vec model had an accuracy of about 70% while Sentence-BERT improved the accuracy to 74%. Sentence-BERT also had a higher weighted F1-score of 0.739 compared to 0.691 for Word2Vec. Sentence-BERT correctly classified more AI, big-tech, and programming comments, although Word2Vec correctly classified slightly more comments in the other category. Both models had the most difficulty with the programming category which may be because it had much fewer examples than the other classes. Sentence-BERT performed better because its contextual embeddings were able to capture more information about the meaning of the comments.